# SolarScan — 06 · Modèle final, analyse d'erreurs & sauvegarde

Trois objectifs :
1. **Entraîner le modèle final** (recette du 04b) une bonne fois.
2. **Analyser ses erreurs** — comprendre *où* et *pourquoi* il se trompe (pas juste le score).
3. **Sauvegarder** le modèle (`.pt` + ONNX) pour la démo et le déploiement.

> ⚡ Colab GPU, Run all. ~15 min.

## 0. Setup

In [ ]:
%pip install -q torch torchvision scikit-learn pandas pillow matplotlib onnx

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', device)

In [ ]:
DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
SPLITS = Path('splits') if Path('splits').exists() else Path('../splits')

if not (DATA_DIR / 'module_metadata.json').exists():
    import urllib.request, zipfile, shutil
    url = 'https://github.com/RaptorMaps/InfraredSolarModules/raw/master/2020-02-14_InfraredSolarModules.zip'
    urllib.request.urlretrieve(url, 'solar.zip')
    with zipfile.ZipFile('solar.zip') as z:
        z.extractall('tmp')
    shutil.move('tmp/InfraredSolarModules', 'data')
    DATA_DIR = Path('data')

if not (SPLITS / 'train.csv').exists():
    from sklearn.model_selection import train_test_split
    with open(DATA_DIR / 'module_metadata.json', encoding='utf-8') as f:
        meta = json.load(f)
    data = pd.DataFrame([{'filepath': v['image_filepath'], 'classe': v['anomaly_class']}
                         for v in meta.values()])
    tr, tmp = train_test_split(data, test_size=0.30, stratify=data['classe'], random_state=42)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp['classe'], random_state=42)
    SPLITS = Path('splits'); SPLITS.mkdir(exist_ok=True)
    tr.to_csv(SPLITS / 'train.csv', index=False)
    va.to_csv(SPLITS / 'val.csv', index=False)
    te.to_csv(SPLITS / 'test.csv', index=False)
print('OK')

## 1. Données

In [ ]:
CLASSES = sorted(pd.read_csv(SPLITS / 'train.csv')['classe'].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class SolarDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path); self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return self.transform(Image.open(DATA_DIR / row['filepath'])), CLASS_TO_IDX[row['classe']]

train_dl = DataLoader(SolarDataset(SPLITS / 'train.csv', train_tf), batch_size=64, shuffle=True, num_workers=2)
val_dl = DataLoader(SolarDataset(SPLITS / 'val.csv', eval_tf), batch_size=64, num_workers=2)
test_dl = DataLoader(SolarDataset(SPLITS / 'test.csv', eval_tf), batch_size=64, num_workers=2)
n_train = len(train_dl.dataset)
print('train/val/test :', n_train, len(val_dl.dataset), len(test_dl.dataset))

## 2. Entraînement du modèle final (ResNet-18, 2 phases)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

y_train = pd.read_csv(SPLITS / 'train.csv')['classe'].map(CLASS_TO_IDX).values
w = np.sqrt(compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float).to(device), label_smoothing=0.1)

def macro_f1(model, loader):
    model.eval(); preds, gts = [], []
    with torch.no_grad():
        for x, y in loader:
            preds += model(x.to(device)).argmax(1).cpu().tolist(); gts += y.tolist()
    return f1_score(gts, preds, average='macro'), preds, gts

def train(optimizer, scheduler, epochs, tag):
    best, best_state, wait = 0.0, None, 0
    for epoch in range(epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(); criterion(model(x), y).backward(); optimizer.step()
        if scheduler is not None:
            scheduler.step()
        f1, _, _ = macro_f1(model, val_dl)
        print(f'[{tag}] epoch {epoch+1}/{epochs} - val macro-F1 {f1:.3f}')
        if f1 > best:
            best, best_state, wait = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 5:
                break
    if best_state:
        model.load_state_dict(best_state)
    return best

# Phase 1 : tete seule
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
train(torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4), None, 3, 'tete')

# Phase 2 : fine-tuning complet
for p in model.parameters():
    p.requires_grad = True
opt2 = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=12)
best = train(opt2, sched, 12, 'fine-tune')
print('Meilleur val macro-F1 :', round(best, 3))

## 3. Évaluation test

In [ ]:
_, preds, gts = macro_f1(model, test_dl)
preds, gts = np.array(preds), np.array(gts)
print('Test accuracy :', round(accuracy_score(gts, preds), 3))
print('Test macro-F1 :', round(f1_score(gts, preds, average='macro'), 3))
print(classification_report(gts, preds, target_names=CLASSES))

## 4. Analyse d'erreurs

### 4.1 Matrice de confusion normalisée (par ligne = rappel)
Chaque ligne somme à 1 : on lit directement *quelle fraction* d'une vraie classe va vers chaque prédiction.

In [ ]:
cm = confusion_matrix(gts, preds)
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cmn, display_labels=CLASSES).plot(ax=ax, xticks_rotation=90, values_format='.2f')
plt.title('Matrice de confusion normalisee (rappel par classe)')
plt.tight_layout()
plt.show()

### 4.2 Les confusions les plus fréquentes

In [ ]:
pairs = []
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], CLASSES[i], CLASSES[j]))
pairs.sort(reverse=True)
print('Top confusions (vraie -> predite) :')
for n, t, p in pairs[:10]:
    print(f'  {t:16s} -> {p:16s} : {n}')

### 4.3 Galerie d'images mal classées

In [ ]:
test_df = pd.read_csv(SPLITS / 'test.csv').reset_index(drop=True)
wrong = np.where(gts != preds)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, idx in zip(axes.ravel(), wrong):
    row = test_df.iloc[idx]
    ax.imshow(Image.open(DATA_DIR / row['filepath']).convert('L'), cmap='inferno')
    ax.set_title('vrai ' + CLASSES[gts[idx]] + ' / pred ' + CLASSES[preds[idx]], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Sauvegarde du modèle (déploiement)

On enregistre les **poids** (`.pt`), la **liste des classes** (`.json`) et un **export ONNX** (format portable pour la production). Puis on télécharge ces fichiers.

In [ ]:
torch.save(model.state_dict(), 'solarscan_resnet18.pt')
with open('classes.json', 'w') as f:
    json.dump(CLASSES, f)

model.eval()
dummy = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(model, dummy, 'solarscan_resnet18.onnx',
                  input_names=['input'], output_names=['logits'], opset_version=12)
print('Sauvegarde : solarscan_resnet18.pt + classes.json + solarscan_resnet18.onnx')

try:
    from google.colab import files
    files.download('solarscan_resnet18.pt')
    files.download('classes.json')
except Exception:
    print('(Hors Colab : recupere les fichiers manuellement.)')

## 6. Bilan

- Modèle final entraîné + **analyse d'erreurs** (confusions, images ratées).
- Poids **sauvegardés** (`.pt`, ONNX) → réutilisables sans réentraîner.

**À faire ensuite :** récupère `solarscan_resnet18.pt` et `classes.json` (téléchargés par la dernière cellule) et place-les dans `models/` du projet. Ils alimenteront la **démo Gradio** (notebook 07).

➡️ Observe les *top confusions* : souvent Hot-Spot ↔ Hot-Spot-Multi, ou des anomalies discrètes ↔ No-Anomaly. C'est cohérent avec l'EDA (classes proches visuellement).